# Model 2: Temporal Poisson model

Non-homogeneous temporal Poisson model. 
Here we assume the rate of the crimes depend on both the hour of the day and the day of the week.

In [13]:
import numpy as np
import pandas as pd
import torch

import pyro
import pyro.distributions as dist
from pyro.contrib.autoguide import AutoDiagonalNormal
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.optim import Adam, ClippedAdam

In [14]:
# Import helpers
import sys
sys.path.append("../..")

from src.config import cfg
from src.evaluation import compute_error, log_results

# Fetch global params from config
train_perc = cfg.train_size
n_steps = cfg.n_steps
lr = cfg.lr

num_samples = cfg.num_samples

In [15]:
# Start by loading the aggregated dataframe
df = pd.read_csv('../../data/processed/pd_2018_present_agg.csv')
df = df.sort_values(["year", "month", "day", "hour"]).reset_index(drop=True)  # sort the data to make sure train test split is correct
df.head()

,year,month,day,weekday,hour,district,incident_count
0,2018,1,1,0,0,Bayview,16
1,2018,1,1,0,0,Central,18
2,2018,1,1,0,0,Ingleside,6
3,2018,1,1,0,0,Mission,10
4,2018,1,1,0,0,Northern,21


In [16]:
# Decide what data to use
# For Model 2 we want to look at day of week and time of day

# TODO: CHOOSE HOW WE IMPLEMENT
hour = df["hour"].values.astype(int)       # 0..23
weekday = df["weekday"].values.astype(int) # 0..6

# So here we'd be looking at hour of the week
time_idx = (24*weekday + hour).astype(int)

y = df['incident_count'].values

In [17]:
# Split the data into train and test
# Here we dont do random but in chronological order because temporal data
# TODO: SHOULD WE INSERT A TEMPORAL GAP TO AVOID LEAKAGE
split_point = int(train_perc*len(y))

X_train = time_idx[:split_point]
X_test = time_idx[split_point:]

y_train = y[:split_point]
y_test = y[split_point:]


print("num train: %d" % len(y_train))
print("num test: %d" % len(y_test))

num train: 287845
num test: 148284


In [18]:
# Change into tensors
X_train_torch = torch.tensor(X_train, dtype=torch.long)
X_test_torch = torch.tensor(X_test, dtype=torch.long)

y_train_torch = torch.tensor(y_train, dtype=torch.float32)
y_test_np = y_test.astype(float)

n_test = len(y_test_np)

In [19]:
def temporal_poisson_model(X, y=None):

    log_rates = pyro.sample(
        "log_rates",
        dist.Normal(
            torch.zeros(168),
            2 * torch.ones(168)
        ).to_event(1)
    )

    rate = torch.exp(log_rates[X])

    with pyro.plate("data", len(X)):
        pyro.sample("obs", dist.Poisson(rate), obs=y)

In [20]:
# Now we train the model
pyro.clear_param_store()

guide = AutoDiagonalNormal(temporal_poisson_model)

optimizer = ClippedAdam({"lr": lr})
svi = SVI(temporal_poisson_model, guide, optimizer, loss=Trace_ELBO())

for step in range(n_steps):
    loss = svi.step(X_train_torch, y_train_torch)
    
    if step % 500 == 0:
        print(f"[{step}] ELBO loss: {loss:.2f}")

[0] ELBO loss: 768999.78
[500] ELBO loss: 514162.10
[1000] ELBO loss: 513230.26
[1500] ELBO loss: 513212.71
[2000] ELBO loss: 513203.41
[2500] ELBO loss: 513200.60


In [21]:
predictive = Predictive(
    temporal_poisson_model,
    guide=guide,
    num_samples=num_samples,
    return_sites=("log_rates",)
)

samples = predictive(X_test_torch)

log_rates = samples["log_rates"].squeeze(1)   # [S, 168]
rates = torch.exp(log_rates)                  # [S, 168]

preds = rates[:, X_test_torch]                # [S, N_test]

y_pred = preds.mean(dim=0).detach().numpy()

In [22]:
corr, mae, rmse, r2 = compute_error(y_test_np, y_pred)

print("\nTEMPORAL POISSON MODEL")
print(f"First prediction: {y_pred[0]:.3f}")
print(f"CorrCoef: {corr:.3f}")
print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")


TEMPORAL POISSON MODEL
First prediction: 1.959
CorrCoef: 0.171
MAE: 1.207
RMSE: 1.607
R2: 0.016


In [24]:
# Log results in the model_results.csv
log_results("temporal_poisson_model", corr, mae, rmse, r2)

NameError: name 'path' is not defined